# T539 Closed-Loop AOS State — infocus_initial_alignment (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-07-08  
**Last Modified:** 2026-07-08  
**Status:** In Progress  
**Keywords:** Rubin Observatory, AOS, closed loop, MTAOS, wavefront, degree of freedom, T539

## Description

Locate the in-focus images taken at the end of BLOCK-T539 closed-loop alignment
sequences and collect their delivered image quality and AOS state.

Selection criteria:
- `science_program` LIKE `BLOCK-T539%`
- `observation_reason` == `infocus_initial_alignment`
- observation annotation (`scheduler_note`) contains `closed_loop_22dof_trunc12`
- `day_obs >= day_obs_min`

For each matched image we collect:
1. **PSF FWHM median** — ConsDB `visit1_quicklook.psf_sigma_median`
2. **Donut blur FWHM** and **residual AOS FWHM** — ConsDB quicklook (if present)
3. **Aggregated Degree-of-Freedom vector** (total applied trim / offset from LUT) — EFD `MTAOS.logevent_degreeOfFreedom`
4. **Retrieved wavefront (OPD)** per corner WFS — EFD `MTAOS.logevent_wavefrontError`, Zernikes Z4..Z26 excluding Z20, Z21

**Output:** `output/t539_closedloop_aos_{day_obs_min}_{day_obs_max}.parquet`

**Based on:** locate_test_blocks.ipynb (ConsDB access), intrinsics_lib EFD patterns

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-08 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs_min = 20260420
day_obs_max = 20260708          # bump to today when re-running

instrument = 'lsstcam'
science_program_like = 'BLOCK-T539%'
observation_reason = 'infocus_initial_alignment'
annotation_contains = 'closed_loop_22dof_trunc12'

consdb_url = 'http://consdb-pq.consdb:8080/consdb'

# Corner wavefront sensors (SW0 half-chips): detector id -> raft name
corners = {191: 'R00_SW0', 195: 'R04_SW0', 199: 'R40_SW0', 203: 'R44_SW0'}

# Zernikes to keep: Z4..Z26 excluding Z20, Z21 (Noll) — 21 coefficients
zk_noll = [z for z in range(4, 27) if z not in (20, 21)]

# EFD time padding around each exposure (seconds)
efd_pad = 45.0

output_dir = 'output'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

os.environ["no_proxy"] += ",.consdb"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.time import Time, TimeDelta

from lsst.summit.utils import ConsDbClient
from lsst.summit.utils.efdUtils import makeEfdClient

from tqdm.notebook import tqdm

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [ ]:
async def get_aos_state(efd, t_start, t_end, pad=efd_pad):
    """Fetch aggregated DOF and per-corner wavefront Zernikes around a visit.

    Parameters
    ----------
    efd : EfdClient
    t_start, t_end : str or astropy.time.Time
        Exposure start/end (TAI).
    pad : float
        Seconds of padding on each side of the window.

    Returns
    -------
    dict
        `dof0..dof49` (aggregated / total applied trim) and
        `z{noll}_{corner}` for each corner and each Zernike in `zk_noll`.
    """
    begin = Time(t_start, scale='tai') - TimeDelta(pad, format='sec')
    end = Time(t_end, scale='tai') + TimeDelta(pad, format='sec')
    out = {}

    # --- Aggregated DOF: total applied trim (offset from LUT), 50-vector ---
    dof = await efd.select_time_series(
        "lsst.sal.MTAOS.logevent_degreeOfFreedom", "*", begin.utc, end.utc)
    if len(dof) > 0:
        last = dof.iloc[-1]
        for i in range(50):
            col = f"aggregatedDoF{i}"
            if col in dof.columns:
                out[f"dof{i}"] = last[col]

    # --- Retrieved wavefront (OPD) per corner, Zernikes in nm ---
    wf = await efd.select_time_series(
        "lsst.sal.MTAOS.logevent_wavefrontError", "*", begin.utc, end.utc)
    if len(wf) > 0:
        for det, name in corners.items():
            sub = wf[wf['sensorId'] == det] if 'sensorId' in wf.columns else wf
            if len(sub) == 0:
                continue
            row = sub.iloc[-1]
            for z in zk_noll:
                # Default mapping: contiguous array from Z4 -> index z-4.
                # VERIFY against the actual column list (see Data Access).
                col = f"zernikeEstimated{z - 4}"
                out[f"z{z}_{name}"] = row[col] if col in wf.columns else np.nan
    return out

<a id='data'></a>
## Data Access

In [ ]:
# Select the matching in-focus closed-loop images from ConsDB
client = ConsDbClient(consdb_url)

visits_query = f'''
    SELECT v1.visit_id, v1.day_obs, v1.seq_num, v1.band,
           v1.obs_start, v1.obs_end,
           v1.altitude, v1.azimuth, v1.sky_rotation,
           v1.scheduler_note,
           ql.physical_rotator_angle,
           ql.psf_sigma_median, ql.seeing_zenith_500nm_median
    FROM cdb_{instrument}.visit1 v1
    LEFT JOIN cdb_{instrument}.visit1_quicklook ql
      ON v1.visit_id = ql.visit_id
    WHERE v1.science_program LIKE '{science_program_like}'
      AND v1.observation_reason = '{observation_reason}'
      AND v1.day_obs >= {day_obs_min} AND v1.day_obs <= {day_obs_max}
'''
visits = client.query(visits_query).to_pandas()

# Observation annotation lives in scheduler_note
visits = visits[visits['scheduler_note'].str.contains(
    annotation_contains, na=False)].copy()
visits = visits.sort_values(['day_obs', 'seq_num']).reset_index(drop=True)

# PSF FWHM median (arcsec) from psf_sigma_median (pixels, 0.2"/pix)
visits['psf_fwhm_median'] = 2.355 * visits['psf_sigma_median'] * 0.2

print(f"Matched {len(visits)} images")
visits[['day_obs', 'seq_num', 'band', 'psf_fwhm_median', 'scheduler_note']].head(20)

In [ ]:
# Donut blur + residual AOS FWHM (may or may not be in the quicklook schema)
if len(visits) > 0:
    ids = ",".join(str(v) for v in visits['visit_id'])
    try:
        q2 = f'''SELECT visit_id, donut_blur_fwhm, aos_fwhm
                 FROM cdb_{instrument}.visit1_quicklook
                 WHERE visit_id IN ({ids})'''
        blur = client.query(q2).to_pandas()
        visits = visits.merge(blur, on='visit_id', how='left')
        print("Added donut_blur_fwhm, aos_fwhm")
    except Exception as e:
        print(f"donut_blur_fwhm/aos_fwhm not available in ConsDB: {e}")

In [ ]:
# ---- Inspect EFD field naming before trusting the mappings ----
# Run once; paste the output back if the field names differ from assumptions.
efd = makeEfdClient()

t0 = Time(str(day_obs_min), format='isot') if False else Time('2026-04-20T00:00:00')
t1 = Time('2026-04-30T00:00:00')

dof_probe = await efd.select_time_series(
    "lsst.sal.MTAOS.logevent_degreeOfFreedom", "*", t0.utc, t1.utc)
print("degreeOfFreedom columns:")
print([c for c in dof_probe.columns if 'oF' in c or 'DoF' in c or 'dof' in c][:10], '...')
print(f"  ({len(dof_probe)} rows in probe window)\n")

wf_probe = await efd.select_time_series(
    "lsst.sal.MTAOS.logevent_wavefrontError", "*", t0.utc, t1.utc)
print("wavefrontError zernike columns:")
print([c for c in wf_probe.columns if 'ernike' in c])
print("\nother wavefrontError columns:")
print([c for c in wf_probe.columns if 'ernike' not in c])

<a id='analysis'></a>
## Analysis

In [ ]:
# Loop over matched visits, gather AOS state from the EFD
rows = []
for _, v in tqdm(visits.iterrows(), total=len(visits), desc="EFD per visit"):
    state = await get_aos_state(efd, v['obs_start'], v['obs_end'])
    rows.append({**v.to_dict(), **state})

result = pd.DataFrame(rows)
print(f"Collected {len(result)} rows x {result.shape[1]} cols")
n_dof = len([c for c in result.columns if c.startswith('dof')])
n_zk = len([c for c in result.columns if c.startswith('z') and '_' in c])
print(f"  DOF columns: {n_dof}, Zernike columns: {n_zk}")

<a id='results'></a>
## Results & Plots

In [ ]:
# Summary of delivered image quality per visit
show = ['day_obs', 'seq_num', 'band', 'psf_fwhm_median',
        'donut_blur_fwhm', 'aos_fwhm', 'altitude', 'physical_rotator_angle']
show = [c for c in show if c in result.columns]
result[show]

In [ ]:
# Write full table (metadata + IQ + DOF + per-corner Zernikes) to parquet
os.makedirs(output_dir, exist_ok=True)
out_file = os.path.join(
    output_dir, f't539_closedloop_aos_{day_obs_min}_{day_obs_max}.parquet')
result.to_parquet(out_file, index=False)
print(f"Wrote {len(result)} rows x {result.shape[1]} cols to {out_file}")